# Notebook 3.0: BERTopic Modeling & HDBSCAN Hyperparameter Sweep

**Research Question 2 (RQ2) — Career Topic Evolution Across Regimes**
This notebook:
1. Loads sampled submission titles (concise and clean for clustering).
2. Applies the text cleaning pipeline ().
3. Computes dense sentence embeddings using  (with caching via ).
4. Performs a grid sweep over HDBSCAN  and  (resolving Bug 3).
5. Fits the optimal BERTopic model with precomputed embeddings (preventing double embeddings - Bug 4 fix).
6. Uses  to assign informative labels (Bug 2 fix).
7. Visualizes topics in a 2D UMAP scatter projection and saves the model.

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
from src.modeling import set_seed
from src.modeling.load import load_samples
from src.modeling.text import apply_pipeline
from src.modeling.cache import load_embeddings, save_embeddings, save_topic_model
from src.modeling.topics import fit_topic_model, topic_name_map, run_hdbscan_sweep

set_seed(42)

config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Modeling configuration loaded.')

## Step 1: Load Submission Titles
Extract titles for topic modeling. (Submission titles cluster into cohesive themes without the noise of long body text).

In [ ]:
samples_dir = project_root / 'data' / 'output' / 'samples' / '2.0'
# Fallback to 1.2 if 2.0 has not yet been populated
if not samples_dir.exists() or not list(samples_dir.glob('*.json')):
    samples_dir = (project_root / cfg['paths']['data_samples_dir'] / '1.2').resolve()

df_subs = load_samples(
    samples_dir=samples_dir,
    subreddits=cfg['analysis']['subreddits'],
    dataset_types=['submissions'],
)

# Extract and clean titles
df_subs['cleaned_text'] = apply_pipeline(df_subs['title'])
df_subs = df_subs[df_subs['cleaned_text'].str.strip().str.len() > 5].reset_index(drop=True)
print(f'Ready for topic modeling: {len(df_subs):,} cleaned submission titles.')

## Step 2: Compute or Load Cached Embeddings
Precompute sentence embeddings once with  and cache to disk (Bug 4 fix).

In [ ]:
cache_dir = project_root / cfg['modeling']['cache_dir']
embed_key = f'submissions_all_{len(df_subs)}'

embeddings = load_embeddings(embed_key, cache_dir=cache_dir)
if embeddings is None:
    print('Computing embeddings with SentenceTransformer...')
    embedder = SentenceTransformer(cfg['modeling']['embedding_model'])
    embeddings = embedder.encode(
        df_subs['cleaned_text'].tolist(),
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    save_embeddings(embeddings, embed_key, cache_dir=cache_dir)
else:
    print(f'Loaded cached embeddings with shape: {embeddings.shape}')

## Step 3: HDBSCAN Hyperparameter Sweep
Perform systematic sweep over cluster sizes to minimize outlier rate (Bug 3 fix) and maximize topic coherence.

In [ ]:
umap_params = cfg['modeling']['umap']
hdbscan_base = {
    'metric': cfg['modeling']['hdbscan']['metric'],
    'cluster_selection_method': cfg['modeling']['hdbscan']['cluster_selection_method'],
    'prediction_data': cfg['modeling']['hdbscan']['prediction_data'],
}

exp_dir = project_root / cfg['modeling']['experiment_log_dir']

# Run parameter sweep
sweep_results = run_hdbscan_sweep(
    docs=df_subs['cleaned_text'].tolist(),
    embeddings=embeddings,
    umap_params=umap_params,
    hdbscan_base=hdbscan_base,
    min_cluster_sizes=[15, 25, 35, 50],
    min_samples_list=[5, 10, 15],
    seed=cfg['modeling']['seed'],
    experiment_log_dir=exp_dir,
)

print('--- Sweep Results Summary ---')
print(sweep_results.sort_values(by=['outlier_pct', 'coherence_cv'], ascending=[True, False]))

## Step 4: Fit Final Optimal Model
Select the optimal configuration with outlier rate < 20% and save the model.

In [ ]:
best_hdbscan_params = {
    **hdbscan_base,
    'min_cluster_size': cfg['modeling']['hdbscan']['min_cluster_size'],
    'min_samples': cfg['modeling']['hdbscan']['min_samples'],
}

print(f'Fitting final model with params: {best_hdbscan_params}')
topic_model, topics = fit_topic_model(
    docs=df_subs['cleaned_text'].tolist(),
    embeddings=embeddings,
    umap_params=umap_params,
    hdbscan_params=best_hdbscan_params,
    top_n_words=cfg['modeling']['top_n_words'],
    seed=cfg['modeling']['seed'],
)

df_subs['topic'] = topics
name_map = topic_name_map(topic_model)
df_subs['topic_name'] = df_subs['topic'].map(name_map)

# Save fitted model
save_topic_model(topic_model, key='final_submissions_model', cache_dir=cache_dir)

print('Top 10 Discovered Topics:')
for tid, name in list(name_map.items())[:11]:
    count = (topics == tid).sum()
    print(f'  [{count:5d} docs] {name}')

## Step 5: Visualize Topic Clusters (2D UMAP Projection)
Scatter plot showing topic clusters with corrected legend names (Bug 2 fix).

In [ ]:
# Reduce embeddings to 2D for plotting
from umap import UMAP

umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.1, metric='cosine', random_state=42)
coords_2d = umap_2d.fit_transform(embeddings)

df_subs['x'] = coords_2d[:, 0]
df_subs['y'] = coords_2d[:, 1]

plt.figure(figsize=(12, 8))
# Filter out outliers for clearer visualization of topical clusters
df_plot = df_subs[df_subs['topic'] != -1]
top_topics = df_plot['topic'].value_counts().head(8).index

sns.scatterplot(
    data=df_plot[df_plot['topic'].isin(top_topics)],
    x='x', y='y',
    hue='topic_name',
    palette='tab10',
    alpha=0.6,
    s=15,
)

plt.title('2D UMAP Projection of Reddit Career Submission Topics', fontsize=14, pad=12)
plt.xlabel('UMAP Dimension 1')
plt.ylabel('UMAP Dimension 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Discovered Topics')
plt.tight_layout()

fig_path = project_root / 'data' / 'output' / 'figures' / 'topic_scatter.png'
plt.savefig(fig_path, dpi=300)
plt.show()
print(f'Topic scatter plot saved to: {fig_path}')